In [ ]:
import torch
import numpy as np
import random
from sklearn.model_selection import train_test_split

# =========================================================
# Classe d'augmentation (inchangée)
# =========================================================
class CSIAugmentation:
    def __init__(self, noise_std=0.05, jitter_max=10, 
                 scale_range=(0.95, 1.05), subcarrier_dropout=0.05):
        self.noise_std = noise_std
        self.jitter_max = jitter_max
        self.scale_range = scale_range
        self.subcarrier_dropout = subcarrier_dropout

    def __call__(self, data_np):
        if self.noise_std > 0:
            data_np += np.random.normal(0, self.noise_std, data_np.shape)
        if self.scale_range:
            data_np *= np.random.uniform(*self.scale_range)
        if self.jitter_max > 0:
            shift = np.random.randint(-self.jitter_max, self.jitter_max+1)
            if shift != 0:
                data_np = np.roll(data_np, shift, axis=1)
        if self.subcarrier_dropout > 0:
            n_sub = data_np.shape[0]
            n_drop = int(n_sub * self.subcarrier_dropout)
            if n_drop > 0:
                drop_idx = np.random.choice(n_sub, n_drop, replace=False)
                data_np[drop_idx, :] = 0
        return data_np

# =========================================================
# Fonction d'augmentation sur un ensemble
# =========================================================
def augment_dataset(X, y, augmentation, extra_percent=0):
    """
    X: tenseur (N, 1, 342, 500)
    y: tenseur (N,)
    Retourne (X_aug, y_aug) avec duplication augmentée selon extra_percent.
    """
    X_np = X.squeeze(1).numpy()  # (N, 342, 500)
    y_np = y.numpy()

    n_extra = int(len(X_np) * extra_percent / 100)
    indices_extra = set(random.sample(range(len(X_np)), n_extra)) if n_extra > 0 else set()

    X_aug = []
    y_aug = []

    for i, (x, label) in enumerate(zip(X_np, y_np)):
        # exemplaire original (non augmenté)
        X_aug.append(torch.from_numpy(x).unsqueeze(0))
        y_aug.append(label)
        # exemplaire augmenté (si l'indice est tiré)
        if i in indices_extra:
            x_aug = augmentation(x.copy())
            X_aug.append(torch.from_numpy(x_aug).unsqueeze(0))
            y_aug.append(label)

    X_new = torch.cat(X_aug, dim=0)
    y_new = torch.tensor(y_aug, dtype=torch.long)
    return X_new, y_new

# =========================================================
# 1. Chargement des données prétraitées (sans split)
# =========================================================
X_all_train, y_all_train = torch.load("processed_data/preprocessed_all_train.pt")
X_test, y_test = torch.load("processed_data/preprocessed_test.pt")

print(f"Toutes les données d'entraînement : {X_all_train.shape}")
print(f"Test : {X_test.shape}")

# =========================================================
# 2. Split train / validation (80/20 par exemple)
# =========================================================
train_ratio = 0.8
X_train, X_val, y_train, y_val = train_test_split(
    X_all_train, y_all_train,
    test_size=1 - train_ratio,
    stratify=y_all_train,
    random_state=42
)

print(f"Train (avant augmentation) : {X_train.shape}")
print(f"Validation : {X_val.shape}")

# =========================================================
# 3. Application de l'augmentation UNIQUEMENT sur le train
# =========================================================
aug = CSIAugmentation(noise_std=0.05, jitter_max=10,
                      scale_range=(0.95, 1.05), subcarrier_dropout=0.05)
EXTRA_PERCENT = 20   # à modifier selon besoin

X_train_aug, y_train_aug = augment_dataset(X_train, y_train, aug, extra_percent=EXTRA_PERCENT)

print(f"Train après augmentation (+{EXTRA_PERCENT}%) : {X_train_aug.shape}")

# =========================================================
# 4. Sauvegarde de trois fichiers séparés
# =========================================================
os.makedirs("processed_data", exist_ok=True)
torch.save((X_train_aug, y_train_aug), "processed_data/train.pt")
torch.save((X_val, y_val), "processed_data/val.pt")
torch.save((X_test, y_test), "processed_data/test.pt")

print("✅ Sauvegarde terminée :")
print("   - train.pt  (ensemble d'entraînement augmenté)")
print("   - val.pt    (ensemble de validation, non augmenté)")
print("   - test.pt   (ensemble de test, non augmenté)")

C:\Users\ilyas\AppData\Local\Temp\ipykernel_23008\3865232511.py:62: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  X_train, y_train, X_val, y_val = torch.load("processed_data

Train : torch.Size([748, 1, 342, 500]), torch.Size([748])
Validation : torch.Size([188, 1, 342, 500]), torch.Size([188])
Test : torch.Size([264, 1, 342, 500]), torch.Size([264])
Train original : torch.Size([748, 1, 342, 500]) -> après augmentation : torch.Size([897, 342, 500])
Validation : torch.Size([188, 1, 342, 500])
Test : torch.Size([264, 1, 342, 500])
Sauvegarde terminée avec augmentation.
